# Hidden Markov Model
Hidden Markov Models (HMMs) contain hidden states that we are trying to infer from observed data. This is useful in bioinformatics, because the observed data is what we can directly measure, like sequenced DNA. On the other hand, the hidden states represent the underlying biological context we are trying to uncover or infer. We will use the Viterbi algorithm to find the most likely sequence of hidden states given a sequence of observations.

## Viterbi Algorithm

**Key Features**

* **Optimal Path Finding:** Identifies the most probable sequence of hidden states
* **Dynamic Programming:** Uses efficient tabulation to avoid redundant calculations
* **Log-Space Computation:** Prevents numerical underflow with large sequences
* **Traceback Mechanism:** Reconstructs the optimal state path after computation


**Algorithm Structure**

The Viterbi algorithm involves these key steps:

1. **Initialization:**
    * Set up a probability matrix using initial probabilities and the first observation
    * Initialize traceback matrix for path reconstruction
2. **Recursion:**
    * For each position and possible state, calculate the maximum probability
    * Store both probabilities and traceback pointers
    * Apply transition and emission probabilities at each step
3. **Termination:**
    * Identify the final state with the highest probability
    * Trace back through the matrix to reconstruct the optimal path


**Applications Beyond Sequence Analysis**

While primarily used for biological sequence analysis, this approach has broader applications:

* **Speech Recognition:** Identifying phonemes in audio signals
* **Part-of-Speech Tagging:** Determining grammatical roles in text
* **Gene Finding:** Locating coding regions in DNA sequences
* **Financial Modeling:** Detecting market regimes in time series data


**Computational Considerations**

Important factors to consider in implementation:

* **Time Complexity:**  where  is sequence length and  is number of states
* **Space Complexity:**  for storing the dynamic programming matrix
* **Numerical Stability:** Using log probabilities to prevent underflow
* **Edge Cases:** Handling zero probabilities with pseudocounts

Notes to consider:

* You will only be given these four data structures
* No other template code or coding-by-contract will be provided
* It may benefit you to utlize Object-Oriented Programming (OOP) as you will be developing out the complete HMM suite throughout the HMM modules
* Make no assumptions as to the number of hidden states you will be given
* Make no assumptions as to the number of distinct observations you will be given
* Make no assumptions that the data structures will be modeling CpG islands (these were just examples)

In [1]:
import numpy as np

In [2]:
class HiddenMarkovModel:
    """
    A Hidden Markov Model (HMM) with Viterbi algorithm implementation

    Attributes:
        states (list): The set of possible hidden states in the model.
        initial_probs (dict): Initial state probabilities mapping each state to its probability of being the starting state.
        transition_probs (dict): Transition probabilities mapping each state to a dictionary of probabilities for moving to another state.
        emission_probs (dict): Emission probabilities mapping each state to a dictionary of probabilities for observing each possible output.
    """
    def __init__(self, initial_probs, transition_probs, emission_probs):
        self.states = []
        self.states = list(initial_probs.keys())
        self.initial_probs = initial_probs
        self.transition_probs = transition_probs
        self.emission_probs = emission_probs

    def get_transition_probs(self, state):
        """
        Returns the transition probabilities for a given state.
        :param state: The state whose transition probabilities should be returned.
        :return:
            dict: A dictionary mapping each state to its probability.
        """
        return self.transition_probs[state]

    def get_emission_probs(self, state):
        """
        Returns the emission probabilities for a given state.
        :param state: The state whose emission probabilities should be returned.
        :return:
            dict: A dictionary mapping each state to its probability.
        """
        return self.emission_probs[state]


    def viterbi_algorithm(self, observations):
        """
        Runs the Viterbi algorithm to find the optimal hidden state path for each observation.
        :param observations (list): A list of observations. If one observation sequence is provided, it is wrapped in a list
        :return:
            list: A list of optimal hidden state paths
        """
        if type(observations) != list:
            observations = [observations]

        print(self.states)
        optimal_path = []
        for observation in observations:
            viterbi_matrix, traceback_matrix = self.build_viterbi_traceback_matrix(observation)
            print(f"Observation: {observation}")
            print(f"Viterbi matrix:\n{viterbi_matrix}")
            print(f"Traceback matrix:\n{traceback_matrix}")
            optimal_path_index = self.viterbi_traceback(viterbi_matrix, traceback_matrix)
            print(f"Optimal path index:\n{optimal_path_index}")
            path = self.convert_index_to_states(optimal_path_index)
            print(f"Optimal path:\n{path}\n")
            optimal_path.append(path)


        return optimal_path


    def build_viterbi_traceback_matrix(self, observations):
        """
        Builds the Viterbi and traceback matrices for a given observation sequence.
        :param observations: A single observation
        :return
            tuple:
                - prob_matrix: Matrix of the most likely probabilities at each state and position.
                - traceback_matrix: Matrix storing the previous state indices for reconstructing the optimal path.
        """
        #Initialize viterbi and traceback matrix
        prob_matrix = np.zeros((len(self.states), len(observations)), dtype = float)
        traceback_matrix = np.zeros((len(self.states), len(observations)), dtype = int)

        ####### Iteration ########
        for i, observation in enumerate(observations):
            for j, state in enumerate(self.states):
                # Get initial and emission probabilities for current state
                state_init_probs = self.initial_probs[state]
                state_emit_probs = self.get_emission_probs(state)

                # If we are looking at the first observation
                if i == 0:
                    # Calculate the initial probability per state
                    state_prob = state_init_probs * state_emit_probs[observation] # Come back to you
                    # Use natural log to prevent numerical underflow
                    prob_matrix[j][i] = np.log(state_prob)

                # Else if we are looking at the second observation onward
                else:
                    # Calculate the possible probabilities based on transitioning for all states
                    possible_probs = [np.exp(prob_matrix[k][i-1]) * self.get_transition_probs(prev_state)[state] * state_emit_probs[observation] for k, prev_state in enumerate(self.states)]

                    # Get the max probability and add the log value to the viterbi matrix
                    max_prob = max(possible_probs)
                    prob_matrix[j][i] = np.log(max_prob)

                    # Get the index of the maximum probabilities and add that to the traceback matrix
                    # Theoretically, if there is a tie (although rare as we are dealing with floating point numbers), argmax would choose the first index making it reproducible
                    previous_coords = np.argmax(possible_probs)
                    traceback_matrix[j][i] = previous_coords

        return prob_matrix, traceback_matrix

    def viterbi_traceback(self, viterbi_matrix, traceback_matrix):
        """
        Traces back through the Viterbi and traceback matrices to recover the optimal hidden state path.

        :param viterbi_matrix: Matrix of the most likely probabilities at each state and position.
        traceback_matrix: Matrix storing the previous state indices for reconstructing the optimal path.

        :return:
            list: The optimal hidden state path as a sequence of indices
        """
        # Identify the final state with the highest probability
        end_state = np.argmax(viterbi_matrix[:,-1])

        # Add state to list
        predictions = [int(end_state)]

        # Traceback through the matrix starting at the end of the matrix
        for i in range(len(traceback_matrix[end_state])-1, 0, -1):
            # Append the state index to the predictions
            end_state = traceback_matrix[end_state][i]
            predictions.append(int(end_state))

        # Reverse the predictions so it starts at the beginning of matrix
        return predictions[::-1]

    def convert_index_to_states(self, predictions):
        """
        Converts a list of predicted indices into a list of state names.
        :param predictions (list): The list of indices that make up the optimal path
        :return:
            list: The list of state names in the optimal path
        """
        states_final = []
        for index in predictions:
            states_final.append(self.states[index])
        return states_final

In [4]:
# Example observation sequence following the powerpoint
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.1,
    "G": 0.9
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}

hmm = HiddenMarkovModel(init_probs, trans_probs, emit_probs)
path1 = hmm.viterbi_algorithm(obs)

['I', 'G']
Observation: ACGCGATC
Viterbi matrix:
[[ -4.60517019  -4.24052707  -5.66764343  -7.09475978  -8.52187614
  -11.33528686 -14.14869757 -14.59498468]
 [ -1.02165125  -3.42959686  -5.83754246  -8.24548807 -10.31363561
  -10.3544576  -11.37610885 -13.78405446]]
Traceback matrix:
[[0 1 0 0 0 0 0 1]
 [0 1 1 1 0 0 1 1]]
Optimal path index:
[1, 0, 0, 0, 0, 1, 1, 1]
Optimal path:
['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']



In [353]:
# Example observation sequence
observations = ["GGCACTGAA", "ACGCGATC"]

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "CpG": 0.3,
    "Genome": 0.5,
    "Promoter": 0.2
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "CpG": {"CpG": 0.6, "Promoter": 0.2, "Genome": 0.2},
    "Genome": {"CpG": 0.2, "Promoter": 0.1, "Genome": 0.7},
    "Promoter": {"CpG": 0.1, "Promoter": 0.7, "Genome": 0.2},
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "CpG": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "Genome": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    "Promoter": {"A": 0.2, "C": 0.3, "G": 0.3, "T": 0.2},
}
hmm2 = HiddenMarkovModel(init_probs, trans_probs, emit_probs)
path2 = hmm2.viterbi_algorithm(obs)

['CpG', 'Genome', 'Promoter']
Observation: ACGCGATC
Viterbi matrix:
[[ -3.5065579   -4.42284863  -5.84996498  -7.27708134  -8.7041977
  -11.51760841 -14.33101913 -15.40859555]
 [ -1.89711998  -3.86323284  -5.8293457   -7.79545855  -9.76157141
  -11.32221916 -12.88286691 -14.84897976]
 [ -3.21887582  -4.77952357  -6.34017132  -7.90081907  -9.46146682
  -11.42757967 -13.39369253 -14.95434028]]
Traceback matrix:
[[0 1 0 0 0 0 0 1]
 [0 1 1 1 1 1 1 1]
 [0 2 2 2 2 2 2 2]]
Optimal path index:
[1, 1, 1, 1, 1, 1, 1, 1]
Optimal path:
['Genome', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome']



In [3]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

hmm3 = HiddenMarkovModel(init_probs, trans_probs, emit_probs)
path3 = hmm3.viterbi_algorithm(obs)

['I', 'G']
Observation: GGCACTGAA
Viterbi matrix:
[[ -2.52572864  -3.79869432  -5.07166     -7.73092003  -9.00388571
  -11.66314575 -12.81451921 -15.47377925 -17.22494532]
 [ -1.83258146  -3.54737989  -5.26217832  -6.57151164  -8.28631007
   -9.59564339 -11.31044182 -12.61977514 -13.92910846]]
Traceback matrix:
[[0 0 0 0 0 0 1 0 1]
 [0 1 1 1 1 1 1 1 1]]
Optimal path index:
[1, 1, 1, 1, 1, 1, 1, 1, 1]
Optimal path:
['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']

